In [1]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

In [20]:
from pathlib import Path

import torch
from PIL import Image
import open_clip

from transformers import BlipForConditionalGeneration, BlipProcessor
from bert_score import BERTScorer

def clip_score(image_dir, prompts, batch_size, device=None):
    """
    This is for text-image alignment. The number of prompts must match the number of images in the image_dir. 
    The function will compute CLIP scores for each image-prompt pair.
    
    Compute CLIP scores for a set of images and prompts.

    Args:
        image_dir (str or Path): Directory containing images.
        prompts (list of str): List of text prompts.
        batch_size (int): Number of images to process in a batch.
        device (str, optional): Device to run the model on. Defaults to CUDA if available.

    Returns:
        list of float: CLIP scores for each image-prompt pair.
    """
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    image_dir = Path(image_dir)
    image_paths = sorted(
        path for path in image_dir.iterdir()
        if path.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
    )

    if len(image_paths) != len(prompts):
        raise ValueError(f"Expected one prompt per image, got {len(image_paths)} images and {len(prompts)} prompts.")
    if batch_size < 1:
        raise ValueError("batch_size must be at least 1.")

    model, _, preprocess = open_clip.create_model_and_transforms('ViT-H-14', pretrained='laion2b_s32b_b79k')
    model = model.to(device).eval()
    tokenizer = open_clip.get_tokenizer('ViT-H-14')
    scores = []

    for start in range(0, len(image_paths), batch_size):
        batch_paths = image_paths[start:start + batch_size]
        batch_prompts = prompts[start:start + batch_size]
        images = torch.stack([
            preprocess(Image.open(path).convert("RGB")) for path in batch_paths
        ]).to(device)
        text = tokenizer(batch_prompts).to(device)

        with torch.no_grad():
            autocast_device = "cuda" if device == "cuda" or str(device).startswith("cuda") else "cpu"
            with torch.autocast(autocast_device, enabled=autocast_device == "cuda"):
                image_features = model.encode_image(images)
                text_features = model.encode_text(text)

        image_features /= image_features.norm(dim=-1, keepdim=True)
        text_features /= text_features.norm(dim=-1, keepdim=True)
        batch_scores = (image_features * text_features).sum(dim=-1)
        scores.extend(batch_scores.detach().cpu().tolist())

    return scores

def bert_score(
    image_dir,
    prompts,
    batch_size,
    device=None,
    caption_model="Salesforce/blip-image-captioning-base",
    bert_model_type="microsoft/deberta-large-mnli",
    lang="en",
    max_new_tokens=50,
):
    """
    This is for text-image prompt alignment. The number of prompts must match the number of images in image_dir.
    The function first generates a text prompt/caption for each image, then computes BERTScore F1 against
    the corresponding ground-truth prompt.

    Args:
        image_dir (str or Path): Directory containing images.
        prompts (list of str): Ground-truth text prompts.
        batch_size (int): Number of images and prompt pairs to process in a batch.
        device (str, optional): Device to run the models on. Defaults to CUDA if available.
        caption_model (str, optional): Hugging Face BLIP caption model used to generate prompts.
        bert_model_type (str, optional): Hugging Face model used by BERTScore.
        lang (str, optional): Language code used by BERTScore.
        max_new_tokens (int, optional): Maximum number of tokens generated by the BLIP caption model.

    Returns:
        list of float: BERTScore F1 scores for each generated-prompt/ground-truth-prompt pair.
    """
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    image_dir = Path(image_dir)
    image_paths = sorted(
        path for path in image_dir.iterdir()
        if path.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
    )

    if len(image_paths) != len(prompts):
        raise ValueError(f"Expected one prompt per image, got {len(image_paths)} images and {len(prompts)} prompts.")
    if batch_size < 1:
        raise ValueError("batch_size must be at least 1.")


    processor = BlipProcessor.from_pretrained(caption_model)
    captioner = BlipForConditionalGeneration.from_pretrained(caption_model).to(device).eval()
    scorer = BERTScorer(
        model_type=bert_model_type,
        lang=lang,
        device=device,
        batch_size=batch_size,
        rescale_with_baseline=False,
    )
    scorer._tokenizer.model_max_length = min(scorer._tokenizer.model_max_length, 512)
    scores = []

    for start in range(0, len(image_paths), batch_size):
        batch_paths = image_paths[start:start + batch_size]
        batch_prompts = prompts[start:start + batch_size]
        images = []
        for path in batch_paths:
            with Image.open(path) as image:
                images.append(image.convert("RGB"))

        inputs = processor(images=images, return_tensors="pt", padding=True).to(device)
        with torch.no_grad():
            generated_ids = captioner.generate(**inputs, max_new_tokens=max_new_tokens)
        generated_prompts = processor.batch_decode(generated_ids, skip_special_tokens=True)
        generated_prompts = [prompt.strip() for prompt in generated_prompts]
        print(f"Generated prompts: {generated_prompts}")

        _, _, f1 = scorer.score(generated_prompts, batch_prompts, batch_size=batch_size)
        scores.extend(f1.detach().cpu().tolist())

    return scores

In [21]:
prompts = [
    "a colorful clover field at sunrise, high detail",
    "a close-up photo of a bright green clover leaf with dew",
    "a small robot holding a clover in a clean studio photo",
    "an impressionist painting of clovers under warm sunlight",
]

scores = clip_score("/home/azm0269@auburn.edu/clover/outputs/ddpo", prompts, batch_size=4, device="cpu")


bert_scores = bert_score("/home/azm0269@auburn.edu/clover/outputs/ddpo", prompts, batch_size=4, device="cpu")


scores, bert_scores

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/388 [00:00<?, ?it/s]

[transformers] DebertaModel LOAD REPORT from: microsoft/deberta-large-mnli
Key                 | Status     |  | 
--------------------+------------+--+-
classifier.bias     | UNEXPECTED |  | 
config              | UNEXPECTED |  | 
classifier.weight   | UNEXPECTED |  | 
pooler.dense.bias   | UNEXPECTED |  | 
pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Generated prompts: ['a field of flowers with the sun setting in the background', 'pink flowers in the field at sunset', 'a green leaf with water droplets on it', 'a robot holding a clover with a green leaf']


([0.32558080554008484,
  0.06635494530200958,
  0.13231976330280304,
  0.10255680233240128],
 [0.6495035290718079,
  0.5861673951148987,
  0.5145587921142578,
  0.6112095713615417])